# EDA avanzado con yfinance (interactivo + valores por defecto)

Este notebook:

- descarga de datos con `yfinance`
- entrada por teclado de **tickers**, fechas e intervalo
- **valores por defecto** si no introduces nada
- control básico de errores
- calidad del dato: nulos, duplicados, cobertura temporal y outliers simples
- análisis OHLCV en formato **ancho** y **largo**
- retornos simples y logarítmicos
- volatilidad rolling
- drawdown
- correlaciones
- medias móviles
- rendimiento acumulado normalizado
- resumen final con métricas útiles para EDA financiero

## Qué he mantenido de tu trabajo
He respetado la idea de:
- usar `yfinance` como fuente
- poder elegir tickers por teclado
- convertir a formato largo
- revisar calidad del dato
- estudiar close, returns, volatilidad, drawdown, correlación y medias móviles

## Qué he mejorado
Además, este notebook incluye:
- entrada segura con *fallback* por defecto
- validación de tickers descargados
- resumen por ticker con más métricas
- comparación de series normalizadas a base 100
- distribución de retornos y percentiles extremos
- análisis mensual básico
- top/bottom días
- volatilidad y volumen anómalos
- guardado opcional de CSV por ticker

In [ ]:
# Si lo necesitas:
# !pip install yfinance pandas numpy matplotlib

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)

DATA_DIR = Path("./data")
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("OK - entorno listo")

## 1) Parámetros interactivos con valores por defecto

Este bloque analysis_goala pedir datos por teclado.  
Si ejecutas el notebook en un entorno donde `input()` no funciona bien, seguirá con valores por defecto.


In [ ]:
DEFAULT_TICKERS = ["SPY", "QQQ", "^GSPC", "^VIX"]
DEFAULT_START = "2015-01-01"
DEFAULT_END = None
DEFAULT_INTERVAL = "1d"
DEFAULT_FOCUS = "SPY"
DEFAULT_SAVE_CSV = "n"

def safe_input(prompt: str, default: str | None = None):
    try:
        value = input(prompt)
        if value is None:
            return default
        value = value.strip()
        return value if value != "" else default
    except Exception:
        return default

tickers_in = safe_input(
    f"Introduce tickers separados por coma [{', '.join(DEFAULT_TICKERS)}]: ",
    ", ".join(DEFAULT_TICKERS)
)

TICKERS = [t.strip() for t in tickers_in.split(",") if t.strip()]
START = safe_input(f"START (YYYY-MM-DD) [{DEFAULT_START}]: ", DEFAULT_START)
END = safe_input("END (YYYY-MM-DD) [vacío = hasta hoy]: ", None)
INTERVAL = safe_input(f"INTERVAL (1d,1wk,1mo,1h,...) [{DEFAULT_INTERVAL}]: ", DEFAULT_INTERVAL)
FOCUS_TICKER = safe_input(f"Ticker principal para análisis individual [{DEFAULT_FOCUS}]: ", DEFAULT_FOCUS)
SAVE_CSV = safe_input(f"¿Guardar CSV por ticker? (s/n) [{DEFAULT_SAVE_CSV}]: ", DEFAULT_SAVE_CSV)

print("TICKERS:", TICKERS)
print("START:", START)
print("END:", END)
print("INTERVAL:", INTERVAL)
print("FOCUS_TICKER:", FOCUS_TICKER)
print("SAVE_CSV:", SAVE_CSV)

## 2) Descarga de datos

In [ ]:
raw = yf.download(
    tickers=TICKERS,
    start=START,
    end=END, 
    interval=INTERVAL, # Intervalo temporal por el que buscamos los datos
    group_by="ticker", # Organiza la salida cuando descargas varios tickers
    auto_adjust=False, # TRUE: precios ajustados, FALSE: precios originales
    threads=True, # Poder descargar varios tickers en paralelo
    progress=False # Controla si aparece la barra de progreso en pantalla
)

print("Shape raw:", raw.shape)
raw.tail()

In [ ]:
def available_tickers(raw_df: pd.DataFrame, requested: list[str]) -> list[str]:
    if raw_df.empty:
        return []
    if isinstance(raw_df.columns, pd.MultiIndex):
        found = sorted(set(c[0] for c in raw_df.columns))
        return [t for t in requested if t in found]
    return requested[:1]

VALID_TICKERS = available_tickers(raw, TICKERS)

if not VALID_TICKERS:
    raise ValueError("No se han descargado datos válidos. Revisa los tickers, fechas o intervalo.")

if FOCUS_TICKER not in VALID_TICKERS:
    FOCUS_TICKER = VALID_TICKERS[0]

print("Tickers válidos descargados:", VALID_TICKERS)
print("Ticker principal usado:", FOCUS_TICKER)


## 3) Funciones auxiliares

In [ ]:
def to_long_ohlcv(raw_df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(raw_df.columns, pd.MultiIndex):
        pieces = []
        for t in sorted(set(c[0] for c in raw_df.columns)):
            sub = raw_df[t].copy()
            sub = sub.rename(columns={"Adj Close": "Adj_Close"})
            sub["Ticker"] = t
            pieces.append(sub.reset_index())
        df = pd.concat(pieces, ignore_index=True)
    else:
        df = raw_df.copy().reset_index()
        df = df.rename(columns={"Adj Close": "Adj_Close"})
        df["Ticker"] = VALID_TICKERS[0]
    cols = ["Date", "Ticker"] + [c for c in df.columns if c not in ["Date", "Ticker"]]
    return df[cols]

def get_close_wide(raw_df: pd.DataFrame, tickers: list[str]) -> pd.DataFrame:
    if isinstance(raw_df.columns, pd.MultiIndex):
        close = pd.DataFrame({t: raw_df[(t, "Close")] for t in tickers if (t, "Close") in raw_df.columns})
    else:
        close = raw_df[["Close"]].rename(columns={"Close": tickers[0]})
    return close.sort_index()

def get_volume_wide(raw_df: pd.DataFrame, tickers: list[str]) -> pd.DataFrame:
    if isinstance(raw_df.columns, pd.MultiIndex):
        volume = pd.DataFrame({t: raw_df[(t, "Volume")] for t in tickers if (t, "Volume") in raw_df.columns})
    else:
        volume = raw_df[["Volume"]].rename(columns={"Volume": tickers[0]})
    return volume.sort_index()

def compute_drawdown(price: pd.Series) -> pd.DataFrame:
    s = price.dropna().copy()
    cummax = s.cummax()
    dd = s / cummax - 1.0
    return pd.DataFrame({"price": s, "cummax": cummax, "drawdown": dd})

def data_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    rep = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(3),
        "n_unique": df.nunique(dropna=True),
    })
    return rep.sort_values(["pct_missing", "n_missing"], ascending=False)

def annualization_factor(interval: str) -> int:
    interval = str(interval).lower()
    if interval.endswith("d"):
        return 252
    if interval.endswith("wk"):
        return 52
    if interval.endswith("mo"):
        return 12
    if interval.endswith("h"):
        return 252 * 6
    return 252

def rolling_window(interval: str) -> int:
    interval = str(interval).lower()
    if interval.endswith("d"):
        return 21
    if interval.endswith("wk"):
        return 8
    if interval.endswith("mo"):
        return 6
    if interval.endswith("h"):
        return 24
    return 21

def save_raw_per_ticker(raw_df: pd.DataFrame, tickers: list[str], out_dir: Path, interval: str) -> None:
    if isinstance(raw_df.columns, pd.MultiIndex):
        for t in tickers:
            cols = [c for c in raw_df.columns if c[0] == t]
            if not cols:
                continue
            sub = raw_df[cols].copy()
            sub.columns = [c[1] for c in cols]
            sub.to_csv(out_dir / f"{t.replace('^','_')}_{interval}.csv", index=True)
    else:
        raw_df.to_csv(out_dir / f"{tickers[0].replace('^','_')}_{interval}.csv", index=True)


## 4) Formato largo y primera inspección

In [ ]:
df_long = to_long_ohlcv(raw)
df_long["Date"] = pd.to_datetime(df_long["Date"], errors="coerce")
df_long = df_long.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print("Shape largo:", df_long.shape)
print(df_long.head())
print(df_long.tail())

In [ ]:
close = get_close_wide(raw, VALID_TICKERS)
volume = get_volume_wide(raw, VALID_TICKERS)

ret = close.pct_change()
logret = np.log(close).diff()

ANN_FACTOR = annualization_factor(INTERVAL)
ROLL = rolling_window(INTERVAL)
vol = logret.rolling(ROLL).std() * np.sqrt(ANN_FACTOR)

print("Shape close:", close.shape)
print("Factor anualización:", ANN_FACTOR)
print("Ventana rolling:", ROLL)
close.tail()


## 5) Calidad del dato

In [ ]:
print("Duplicados completos:", int(df_long.duplicated().sum()))
print("Duplicados por (Date, Ticker):", int(df_long.duplicated(subset=["Date", "Ticker"]).sum()))
data_quality_report(df_long)


In [ ]:
coverage = (
    df_long.groupby("Ticker")
    .agg(
        start=("Date", "min"),
        end=("Date", "max"),
        n_obs=("Date", "count"),
        missing_close=("Close", lambda s: int(s.isna().sum())),
        missing_volume=("Volume", lambda s: int(s.isna().sum()) if "Volume" in df_long.columns else np.nan),
    )
)

coverage["years_covered"] = ((coverage["end"] - coverage["start"]).dt.days / 365.25).round(2)
coverage


## 6) Estadística descriptiva por ticker

In [ ]:
numeric_cols = [c for c in df_long.columns if c not in ["Date", "Ticker"]]
desc = (
    df_long
    .groupby("Ticker")[numeric_cols]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
)
desc


## 7) Gráficos básicos de precio y volumen

In [ ]:
plt.figure(figsize=(11, 5))
for t in VALID_TICKERS:
    if t in close.columns:
        plt.plot(close.index, close[t], label=t)
plt.title("Precio de cierre")
plt.xlabel("Fecha")
plt.ylabel("Close")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11, 5))
if FOCUS_TICKER in volume.columns:
    plt.plot(volume.index, volume[FOCUS_TICKER])
    plt.title(f"Volumen - {FOCUS_TICKER}")
    plt.xlabel("Fecha")
    plt.ylabel("Volume")
    plt.tight_layout()
    plt.show()


## 8) Series normalizadas a base 100

Esto permite comparar activos con escalas de precios distintas.


In [ ]:
norm_close = close.divide(close.iloc[0]).mul(100)

plt.figure(figsize=(11, 5))
for t in norm_close.columns:
    plt.plot(norm_close.index, norm_close[t], label=t)
plt.title("Evolución normalizada (base 100)")
plt.xlabel("Fecha")
plt.ylabel("Base 100")
plt.legend()
plt.tight_layout()
plt.show()

norm_close.tail()


## 9) Retornos, distribución y extremos

In [ ]:
ticker_focus = FOCUS_TICKER
vals = logret[ticker_focus].dropna()

fig = plt.figure(figsize=(11, 4))
plt.hist(vals, bins=80)
plt.title(f"Distribución de log-returns - {ticker_focus}")
plt.xlabel("log-return")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

moments = pd.Series({
    "mean": float(vals.mean()),
    "std": float(vals.std()),
    "skew": float(vals.skew()),
    "kurtosis": float(vals.kurtosis()),
    "p01": float(vals.quantile(0.01)),
    "p05": float(vals.quantile(0.05)),
    "median": float(vals.quantile(0.50)),
    "p95": float(vals.quantile(0.95)),
    "p99": float(vals.quantile(0.99)),
})
moments.to_frame("value")


In [ ]:
extreme_days = (
    logret[ticker_focus]
    .dropna()
    .sort_values()
    .to_frame("log_return")
)

print("Peores 10 días:")
display(extreme_days.head(10))

print("Mejores 10 días:")
display(extreme_days.tail(10).sort_values("log_return", ascending=False))


## 10) Volatilidad rolling y drawdown

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(vol.index, vol[ticker_focus])
plt.title(f"Volatilidad rolling ({ROLL}) anualizada - {ticker_focus}")
plt.xlabel("Fecha")
plt.ylabel("Volatilidad")
plt.tight_layout()
plt.show()


In [ ]:
dd = compute_drawdown(close[ticker_focus])

plt.figure(figsize=(11, 4))
plt.plot(dd.index, dd["drawdown"])
plt.title(f"Drawdown - {ticker_focus}")
plt.xlabel("Fecha")
plt.ylabel("Drawdown")
plt.tight_layout()
plt.show()

dd["drawdown"].describe().to_frame("drawdown")


## 11) Correlación entre activos

In [ ]:
corr = logret.dropna().corr()
corr


In [ ]:
plt.figure(figsize=(7, 5))
plt.imshow(corr.values, aspect="auto")
plt.title("Correlación de log-returns")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar()
plt.tight_layout()
plt.show()


## 12) Señal simple de tendencia: medias móviles

In [ ]:
SHORT = 50
LONG = 200

ma_s = close[ticker_focus].rolling(SHORT).mean()
ma_l = close[ticker_focus].rolling(LONG).mean()
signal = (ma_s > ma_l).astype(int)

plt.figure(figsize=(11, 5))
plt.plot(close.index, close[ticker_focus], label="Close")
plt.plot(ma_s.index, ma_s, label=f"MA{SHORT}")
plt.plot(ma_l.index, ma_l, label=f"MA{LONG}")
plt.title(f"Tendencia con medias móviles - {ticker_focus}")
plt.xlabel("Fecha")
plt.ylabel("Precio")
plt.legend()
plt.tight_layout()
plt.show()

signal.value_counts().rename(index={0: "Bearish", 1: "Bullish"}).to_frame("n_periodos")


## 13) Análisis mensual básico

Útil para ver estacionalidad simple y dispersión mensual.


In [ ]:
monthly_ret = close.resample("M").last().pct_change()
monthly_summary = pd.DataFrame({
    "mean_monthly_return": monthly_ret.mean(),
    "std_monthly_return": monthly_ret.std(),
    "best_month": monthly_ret.max(),
    "worst_month": monthly_ret.min(),
    "positive_month_ratio": (monthly_ret > 0).mean()
}).sort_values("mean_monthly_return", ascending=False)

monthly_summary


In [ ]:
if ticker_focus in monthly_ret.columns:
    monthly_focus = monthly_ret[ticker_focus].dropna().to_frame("monthly_return")
    monthly_focus["year"] = monthly_focus.index.year
    monthly_focus["month"] = monthly_focus.index.month

    pivot_month = monthly_focus.pivot_table(index="year", columns="month", values="monthly_return", aggfunc="mean")
    pivot_month


## 14) Volumen y días anómalos

In [ ]:
if ticker_focus in volume.columns:
    vol_focus = volume[ticker_focus].dropna()
    vol_z = (vol_focus - vol_focus.mean()) / vol_focus.std() if vol_focus.std() != 0 else vol_focus * np.nan
    unusual_volume = vol_z[vol_z > 3].sort_values(ascending=False).to_frame("zscore_volume")
    unusual_volume.head(15)


## 15) Tabla resumen final por ticker

In [ ]:
summary_rows = []

for t in close.columns:
    s = close[t].dropna()
    if len(s) < 30:
        continue

    lr = np.log(s).diff().dropna()
    r = s.pct_change().dropna()
    dd_t = compute_drawdown(s)["drawdown"]

    ann_ret = lr.mean() * ANN_FACTOR
    ann_vol = lr.std() * np.sqrt(ANN_FACTOR)
    sharpe_like = ann_ret / ann_vol if ann_vol != 0 else np.nan

    summary_rows.append({
        "Ticker": t,
        "start": s.index.min().date(),
        "end": s.index.max().date(),
        "n_obs": int(len(s)),
        "last_close": float(s.iloc[-1]),
        "total_return": float(s.iloc[-1] / s.iloc[0] - 1),
        "ann_log_return": float(ann_ret),
        "ann_vol": float(ann_vol),
        "sharpe_like": float(sharpe_like) if pd.notna(sharpe_like) else np.nan,
        "max_drawdown": float(dd_t.min()),
        "best_day": float(r.max()) if len(r) else np.nan,
        "worst_day": float(r.min()) if len(r) else np.nan,
        "pct_positive_days": float((r > 0).mean()) if len(r) else np.nan,
        "skew_logret": float(lr.skew()) if len(lr) else np.nan,
        "kurtosis_logret": float(lr.kurtosis()) if len(lr) else np.nan,
    })

summary = pd.DataFrame(summary_rows).sort_values(["sharpe_like", "ann_log_return"], ascending=False)
summary


## 16) Guardado opcional de CSV

Guarda un CSV bruto por ticker si lo has indicado.


In [ ]:
if str(SAVE_CSV).lower() in {"s", "si", "sí", "y", "yes"}:
    save_raw_per_ticker(raw, VALID_TICKERS, RAW_DIR, INTERVAL)
    print("CSV guardados en:", RAW_DIR.resolve())
else:
    print("No se han guardado CSV.")


## 17) Conclusiones guía para interpretar el EDA

Cuando ejecutes el notebook, fíjate especialmente en:

1. **Cobertura temporal y nulos**  
   Si un ticker tiene pocos datos o muchos nulos, puede distorsionar las comparaciones.

2. **Retorno vs volatilidad**  
   No basta con mirar quién sube más; mira también la volatilidad y el drawdown.

3. **Distribución de retornos**  
   La asimetría (*skew*) y curtosis ayudan a ver si hay colas pesadas o eventos extremos.

4. **Correlación**  
   Dos activos muy correlacionados aportan poca diversificación.

5. **Drawdown máximo**  
   Es una métrica muy útil para estudiar periodos de crisis o estrés.

6. **Serie normalizada**  
   Ideal para comparar comportamientos relativos aunque los precios absolutos sean muy distintos.

7. **Meses y días extremos**  
   Ayudan a identificar periodos de régimen, shocks o cambios de mercado.
